In [0]:
CATALOG = "workspace"
DATABASE = "qualite_eau"

from pyspark.sql import functions as F

df_silver = spark.table(f"{CATALOG}.{DATABASE}.silver_qualite_eau")
print(f"Silver : {df_silver.count():,} lignes")

In [0]:
# Table 1 - Conformité par commune et année

gold_conformite_commune = (
    df_silver
    .groupBy("code_commune", "nom_commune", "_departement", "annee")
    .agg(
        F.count("*").alias("nb_prelevements"),
        F.sum(F.col("est_conforme").cast("int")).alias("nb_conformes"),
        F.sum((~F.col("est_conforme")).cast("int")).alias("nb_non_conformes"),
        F.round(F.avg(F.col("est_conforme").cast("int")) * 100, 2).alias("taux_conformite_pct")
    )
    .filter(F.col("annee").isNotNull())
)

(gold_conformite_commune.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{DATABASE}.gold_conformite_commune"))

print(f"✅ gold_conformite_commune : {gold_conformite_commune.count():,} lignes")

In [0]:
# Table 2 - Evolution mensuelle des paramètres

gold_evolution = (
    df_silver
    .filter(F.col("categorie_parametre").isin("chimie", "microbiologie", "radioactivite"))
    .filter(F.col("resultat_numerique").isNotNull())
    .filter(F.col("annee").isNotNull())
    .groupBy("annee", "mois", "libelle_parametre", "categorie_parametre", "_departement")
    .agg(
        F.round(F.avg("resultat_numerique"), 4).alias("valeur_moyenne"),
        F.round(F.expr("percentile_approx(resultat_numerique, 0.5)"), 4).alias("mediane"),
        F.round(F.max("resultat_numerique"), 4).alias("valeur_max"),
        F.count("*").alias("nb_mesures")
    )
)

(gold_evolution.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{DATABASE}.gold_evolution_parametres"))

print(f"✅ gold_evolution_parametres : {gold_evolution.count():,} lignes")

In [0]:
# Table 3 - Top 10 communes les plus et moins conformes

gold_conformite = spark.table(f"{CATALOG}.{DATABASE}.gold_conformite_commune")

# Filtre communes avec assez de prélèvements
gold_filtered = gold_conformite.filter(F.col("nb_prelevements") >= 10)

top_meilleures = (gold_filtered
    .orderBy(F.col("taux_conformite_pct").desc())
    .limit(10))

top_pires = (gold_filtered
    .orderBy(F.col("taux_conformite_pct").asc())
    .limit(10))

(top_meilleures.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{DATABASE}.gold_top10_meilleures"))

(top_pires.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{DATABASE}.gold_top10_pires"))

print("✅ gold_top10_meilleures et gold_top10_pires créées")

In [0]:
# Table 4 - Analyse non-conformités par département

gold_non_conformites = (
    df_silver
    .filter(F.col("est_conforme") == False)
    .filter(F.col("annee").isNotNull())
    .groupBy("_departement", "categorie_parametre", "libelle_parametre", "annee")
    .agg(
        F.count("*").alias("nb_non_conformites"),
        F.countDistinct("code_commune").alias("nb_communes_touchees")
    )
    .orderBy(F.desc("nb_non_conformites"))
)

(gold_non_conformites.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{DATABASE}.gold_non_conformites"))

print(f"✅ gold_non_conformites : {gold_non_conformites.count():,} lignes")

In [0]:
# Vérification toutes les tables Gold

tables = [
    "gold_conformite_commune",
    "gold_evolution_parametres", 
    "gold_top10_meilleures",
    "gold_top10_pires",
    "gold_non_conformites"
]

print("📊 Tables Gold créées :")
for t in tables:
    count = spark.table(f"{CATALOG}.{DATABASE}.{t}").count()
    print(f"  • {t} : {count:,} lignes")